# 16 older layouts (g3b)

Every EirGrid contracted edition since 2015 read row by row. What you should see: 72 editions, each parsed, each within 1 MW of the total it prints (the load also checks the set of refs against the page text). Then the exhibit: who joins and who leaves the transmission queue each year.

In [ ]:
import subprocess

from iegrid import store
from iegrid.checks import tso_history_ties, tso_rows_missing_signed

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: every edition parsed and tied to its printed total

In [ ]:
from iegrid.parse import TIE_MW

rows = tso_history_ties(con)
for sid, as_of, n, parsed, printed in rows:
    print(f"{sid:32} {as_of}  rows {n!s:>4}  parsed {parsed!s:>12}  printed {printed!s:>12}")
missing = [r for r in rows if r[2] is None]
off = [r for r in rows if r[2] is not None and (r[4] is None or abs(r[3] - r[4]) > TIE_MW)]
assert not missing, f"editions not parsed: {missing}"
assert not off, f"editions off their printed total: {off}"
print(f"\n{len(rows)} editions, every one parsed and within {TIE_MW} MW of its printed total")
from iegrid.checks import MAX_BLANK_SIGNED_SHARE

blanks = tso_rows_missing_signed(con)
for sid, as_of, without, n, refs in blanks:
    print(f"no signed month printed: {sid} {as_of} {without} of {n} rows {refs}")
lost = [b for b in blanks if b[2] / b[3] > MAX_BLANK_SIGNED_SHARE]
assert not lost, f"editions where the signed column looks lost: {lost}"
print(f"no edition lost its signed column ({sum(b[2] for b in blanks)} rows the PDF leaves blank)")

## refs relabelled for the same project (not counted as joins or leaves)

In [ ]:
from iegrid import flows

for f in flows.flows(con):
    if f.kind == "relabel":
        print(f"{f.as_of}  {f.other_ref:>7} -> {f.ref:<7} {f.name[:45]:45} {float(f.mw):7.1f} MW")

## exhibit: who joins and who leaves

In [ ]:
from iegrid import exhibits

fig = exhibits.queue_flows(con)
exhibits.save(fig, "g3b_queue_flows")